# Darukaa Adaptive Biodiversity Assessment — General Production Colab

**Purpose.** Run the generalized v1.1.0 assessment engine for a single polygon, KML/KMZ, multi-feature GeoJSON, or a Site Selection handoff containing multiple EMUs.

**Assessment unit:** EMU (Ecological Management Unit). An EMU may be multipart/disconnected. Each EMU is characterized, routed by ecological domain, benchmarked against its own compatible reference population, and retained as an auditable unit before project-level aggregation.

This notebook is intentionally **not ecosystem-specific**. Aquatic-only and terrestrial-only metrics are routed conditionally; mixed projects keep domain-specific reference populations separate.

## 0. Reproducibility and run controls

In [ ]:
REPO_URL = "https://github.com/G-auravSingh/reference-benchmarking.git"
GIT_REF = "main"  # Production/client runs must replace this with a verified commit SHA.
REPO_DIR = "/content/reference-benchmarking"
ADAPTIVE_DIR = f"{REPO_DIR}/darukaa_adaptive_v1.1.0"
GEE_PROJECT = "darukaa-earth-product"
EXPECTED_PACKAGE_VERSION = "1.1.0"
OUTPUT_DIR = "/content/darukaa_adaptive_outputs"
DOMAIN_MODE = "auto"  # auto | terrestrial | aquatic | mixed
INPUT_MODE = "upload"  # upload | direct_path


In [ ]:
import os, shutil, subprocess, pathlib, sys
def run_cmd(cmd, cwd=None):
    print("$", " ".join(cmd)); r=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT); print(r.stdout); r.check_returncode()
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
run_cmd(["git","clone","--depth","1","--branch",GIT_REF,REPO_URL,REPO_DIR])
sys.path.insert(0, ADAPTIVE_DIR)
subprocess.check_call([sys.executable,"-m","pip","install","-q","-e",ADAPTIVE_DIR])
import darukaa_adaptive
print("Package:",darukaa_adaptive.__version__)
assert darukaa_adaptive.__version__ == EXPECTED_PACKAGE_VERSION


## 1. Earth Engine initialization

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
print("Earth Engine initialized:", GEE_PROJECT)


## 2. Input acquisition

Accepted uploads:
- **Site Selection handoff ZIP** containing `tile_manifest.json` and referenced GeoJSON tiles;
- `tile_manifest.json` with its GeoJSON tiles;
- multi-feature `.geojson`;
- `.kml` / `.kmz`;
- single `.geojson` polygon.

For a production Site Selection run, use the formal handoff package rather than relying on filenames alone.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
if not uploaded: raise RuntimeError("No input uploaded.")
allowed={".zip",".json",".geojson",".kml",".kmz"}
inputs=[Path(x) for x in uploaded if Path(x).suffix.lower() in allowed]
if not inputs: raise ValueError(f"No supported input found: {list(uploaded)}")
INPUT_FILE=str(inputs[0])
print("Input:", INPUT_FILE)


## 3. Canonicalize the project into EMUs and inspect routing

In [ ]:
from darukaa_adaptive.inputs import load_project_input
project=load_project_input(INPUT_FILE, domain=DOMAIN_MODE)
print("Project:",project.project_id, project.project_name)
print("EMUs:",len(project.emus))
for e in project.emus:
    print(f"  {e.emu_id}: domain={e.resolved_domain(project.project_domain)}, area={e.area_ha:.3f} ha, parts={getattr(e.geometry,'geoms',[e.geometry]).__len__() if hasattr(e.geometry,'geoms') else 1}")


In [ ]:
assert project.validate()==[]
from darukaa_adaptive.handoff import validate_handoff_manifest
print("Input contract: PASS")


## 4. Load profile and configure output

In [ ]:
from darukaa_adaptive.config import AssessmentConfig
from pathlib import Path
from darukaa_adaptive.reference_profiles import ReferenceSelectionPolicy
profile_name = "aquatic_lake" if DOMAIN_MODE=="aquatic" else "terrestrial" if DOMAIN_MODE=="terrestrial" else "mixed"
PROFILE_PATH=f"{ADAPTIVE_DIR}/profiles/{profile_name}.yaml"
cfg=AssessmentConfig.from_yaml(PROFILE_PATH)
cfg.gee.project_id=GEE_PROJECT
cfg.output_dir=OUTPUT_DIR

# Stage C is intentionally manual and must be entered only after reviewing the
# reference diagnostics printed after a failed automatic reference attempt.
# Leave None for the first run. No KML/CSV reference fallback exists.
MANUAL_REFERENCE_HMI_THRESHOLD = None
cfg.reference.manual_hmi_threshold = MANUAL_REFERENCE_HMI_THRESHOLD

errors=cfg.validate()
if errors: raise ValueError("Configuration validation failed:\n- "+"\n- ".join(errors))
print("Profile:",cfg.profile.name,"version",cfg.profile.version)
print("Reference stages:", [x["stage"] for x in ReferenceSelectionPolicy(cfg).stage_plan()])
print("Manual HMI threshold:",cfg.reference.manual_hmi_threshold)


## 5. Run the complete EMU → reference → benchmark → scoring → reporting pipeline

In [ ]:
from darukaa_adaptive.pipeline import AdaptivePipeline
runner=AdaptivePipeline(cfg)
result=runner.run(INPUT_FILE, domain=DOMAIN_MODE)
print("Completed EMUs:",len(result["emus"]))
print("Output directory:",result["output_dir"])
print("Project overall:",result["project_overall"])
print("\nEMU comparison:")
print(result["emu_comparison"].to_string(index=False) if not result["emu_comparison"].empty else "No scored EMU comparison available.")


## 6. Reference governance and coverage QA

In [ ]:
for r in result["emus"]:
    print("\nEMU",r["emu_id"],"domain",r["emu_domain"])
    print("  area_ha:",round(r["emu_area_ha"],3))
    print("  reference populations:", list(r["reference_populations"]))
    for k,v in r["reference_populations"].items():
        print("  ",k,"status=",v.get("status"),"approval=",v.get("approval"),"state=",v.get("reference_state"))


## 7. Deliverables and audit record

The run directory contains one auditable output tree per EMU plus project-level aggregation tables and `project_assessment_manifest.json`. The manifest records the normalized EMU model, routing, outputs, and aggregation coverage. Missing/insufficient EMUs remain visible through coverage fields and are not silently converted to zero.

In [ ]:
from pathlib import Path
out=Path(result["output_dir"])
for name in ["project_assessment_manifest.json","project_overall_scorecard.json","project_pillar_aggregation.csv","project_metric_score_aggregation.csv","emu_ecological_comparison.csv","Project_Biodiversity_Baseline_Report.html"]:
    print(name, "->", out/name)


## 8. Production release checklist

Before client delivery: pin the exact Git commit; retain the input/handoff archive; retain `project_assessment_manifest.json`; inspect reference governance for every EMU; confirm metric coverage and missing-data status; review proxy labels and uncertainty; verify report tables against CSV outputs; and archive the exact configuration/profile used.